# ESCI semantic hybrid / XGBoost v4

このnotebookでは、これまでのv3 lexical LTRを壊さず、semantic retrieval と semantic特徴量つきXGBoostを追加するv4実験を扱います。

## 方針

1. 商品文書を `intfloat/multilingual-e5-base` でembedding化します。
   - query側: `query: ...`
   - product側: `passage: title | brand: ... | color: ... | bullet: ...`
   - 正規化済みembeddingなので、内積を `semantic_cosine` として使います。
2. OpenSearchには別index `amazon-jp-semantic-v1` を作り、既存のlexical fieldに `product_embedding` を追加します。
3. 候補生成は `bool.should` で `lexical_v2` 相当のBM25クエリと `knn` クエリを同時に使います。
   - RRFではありません。
   - 初段scoreはOpenSearchの通常 `_score` 合成、つまりざっくり `BM25 + beta * kNN_score` 系になります。
4. 最終rankingはv3特徴量 + `semantic_cosine` でXGBoost Rankerを学習します。
5. XGBoost v4のquery内順位を `lexical_v2_norm` と `semantic_norm` で近似するcheap_score回帰を行い、初段候補生成の `beta` 設計に使います。

## 重要な実装メモ

OpenSearch k-NN script自体は、固定JSONでquery vectorを渡すと正常に動作します。一方で、今回のOpenSearch LTR pluginのfeature templateでは、`query_vector` のような配列paramがMustache展開時に文字列化され、`knn_score` が期待する数値配列として渡せませんでした。

そのため、このv4ではsemantic特徴量込みのXGBoostを **OpenSearch LTR plugin内ではなく、外部リランカーとして評価** します。v3 LTR pluginは引き続きOpenSearch内で使えます。productionに寄せるなら、アプリ側でtopN候補に対してv4 XGBoostを適用する構成が現実的です。

## 生成物

- `amazon/semantic_embeddings.py`: semantic index / embedding生成ユーティリティ
- `amazon/run_semantic_hybrid_v4.py`: v4学習・評価・cheap_score回帰
- `amazon/artifacts/semantic_hybrid/`: 商品embedding cache
- `amazon/artifacts/semantic_hybrid_v4/`: v4評価結果


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path('..').resolve()
ART = ROOT / 'amazon' / 'artifacts' / 'semantic_hybrid_v4'
ART

## 1. pilot indexでhybrid DSLの疎通確認

先頭1,000商品のpilot indexでは、`bool.should` にlexical queryとk-NN queryを並べる検索がHTTP 200で動くことを確認済みです。pilotは商品分布が偏っているので、ここでは品質ではなくDSL疎通だけを見ます。

In [ ]:
# 必要なら再実行
# !../.venv/bin/python semantic_embeddings.py all --limit 1000 --index amazon-jp-semantic-pilot --recreate --batch-size 64 --bulk-size 250

import sys, requests
sys.path.insert(0, str(ROOT / 'amazon'))
from semantic_embeddings import encode_queries, BASE_URL
from run_semantic_hybrid_v4 import hybrid_candidate_body

q = '自転車スピーカー'
v = encode_queries([q])[0]
body = hybrid_candidate_body(q, v, size=3, k=10, semantic_boost=1.0)
r = requests.post(f'{BASE_URL}/amazon-jp-semantic-pilot/_search', json=body, timeout=30)
print(r.status_code)
r.json()['hits']['hits'][:3]

## 2. full semantic index作成

CPUのみだと全339,059商品のembedding生成は重めです。`product_embeddings_full.npy` と `product_ids_full.json` ができた後、index作成・bulk ingestを実行します。

In [ ]:
# 全商品embedding生成（時間がかかります）
# !../.venv/bin/python semantic_embeddings.py encode --batch-size 128

# full index作成とingest
# !../.venv/bin/python semantic_embeddings.py create-index --index amazon-jp-semantic-v1 --recreate
# !../.venv/bin/python semantic_embeddings.py ingest --index amazon-jp-semantic-v1 --bulk-size 250
# !../.venv/bin/python semantic_embeddings.py smoke --index amazon-jp-semantic-v1

## 3. v4 XGBoost + cheap_score回帰

v3のfeature parquetを読み込み、`semantic_cosine` を追加してXGBoost Rankerを学習します。full embedding cacheがあればそれを使い、なければESCI judged productだけを別cacheとしてembedding化します。

出力される主なファイル:

- `metrics.json`: v3/v4 official nDCG、paired bootstrap、cheap_score係数
- `validation_tuning.csv`: XGBoost設定別validation結果
- `cheap_score_regression.csv`: cheap_score候補の係数と近似性能
- `feature_importance.csv`: v4特徴量重要度
- `test_predictions.csv`: test pairごとのv3/v4/semantic score

In [ ]:
# 実行本体
# !../.venv/bin/python run_semantic_hybrid_v4.py --batch-size 128 --semantic-index amazon-jp-semantic-v1

metrics_path = ART / 'metrics.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    metrics
else:
    print('metrics.json はまだありません。上のセルを実行してください。')

In [ ]:
for name in ['validation_tuning.csv', 'cheap_score_regression.csv', 'feature_importance.csv']:
    path = ART / name
    if path.exists():
        print('\n###', name)
        display(pd.read_csv(path).head(20))

## 4. 解釈ポイント

- `test_v4_official_nDCG` が `test_v3_official_nDCG` を上回るかを見る。
- `semantic_cosine` のfeature importanceが高いかを見る。
- `cheap_score_selected.beta_semantic_norm` が十分正なら、初段候補生成でsemantic側のboostを強める価値があります。
- `mean_top10_overlap_with_v4` は、cheap_score top10がv4 top10をどれだけ拾えているかの近似指標です。
- full semantic indexがある場合、`hybrid_smoke_top10.csv` で実際のhybrid候補の目視確認も行います。